# Hierarchical clustering and DBSCAN
**Machine Learning · Week 8 (The rest of the classical toolbox) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w8d3.html

**Hierarchical clustering** starts with every point alone and keeps merging the two closest groups.
The whole merge history is a tree, the **dendrogram**. Cut it at any height to get clusters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=30, centers=[[0, 0], [4, 1], [1.5, 4]], cluster_std=0.7, random_state=2)
Z = linkage(X, method="ward")              # each row of Z is one merge: (group a, group b, height, size)
cut = 5.0
labels = fcluster(Z, t=cut, criterion="distance")   # cut the tree at height 5

palette = np.array(["#ff8a50", "#5577f5", "#17a98c", "#e45fa3", "#7b5cfa", "#f0ac2e", "#5aa9f2", "#bcb7cb"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
set_link_color_palette(list(palette[:3]))
tree = dendrogram(Z, ax=ax2, color_threshold=cut, above_threshold_color="#8a85a3")
color_of = dict(zip(tree["leaves"], tree["leaves_color_list"]))     # each point gets its branch's color
ax1.scatter(X[:, 0], X[:, 1], c=[color_of[i] for i in range(len(X))], s=40)
for i, (a, b) in enumerate(X):
    ax1.annotate(str(i), (a, b), fontsize=7, xytext=(3, 3), textcoords="offset points", color="#57526f")
ax1.set_title(f"{labels.max()} clusters from cutting at height {cut}")
ax2.axhline(cut, color="#2c2842", ls="--", lw=1.2)
ax2.set_title("The dendrogram: every merge, and its height")
ax2.set_xlabel("point number")
plt.show()

## What does "closest groups" mean? Four linkages
Between two groups of points you can measure the **single** (nearest pair), **complete** (farthest pair) or
**average** distance, or use **Ward**'s rule: merge the pair that adds the least spread. Single linkage
follows chains, so it can trace the moons; Ward prefers round, even-sized groups.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import AgglomerativeClustering

Xm, _ = make_moons(n_samples=200, noise=0.06, random_state=0)
Xb, _ = make_blobs(n_samples=200, centers=[[0, 0], [4, 0], [2, 3.5]], cluster_std=[0.5, 1.0, 0.7], random_state=5)
fig, axes = plt.subplots(2, 4, figsize=(13, 5.6), layout="constrained")
for col, method in enumerate(["single", "complete", "average", "ward"]):
    for row, data in enumerate([Xm, Xb]):
        k = 2 if row == 0 else 3
        lab = AgglomerativeClustering(n_clusters=k, linkage=method).fit_predict(data)
        axes[row, col].scatter(data[:, 0], data[:, 1], c=palette[lab], s=10)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
    axes[0, col].set_title(f"{method} linkage")
axes[0, 0].set_ylabel("2 moons")
axes[1, 0].set_ylabel("3 blobs")
plt.show()

## DBSCAN: clusters are crowds
Two settings: a radius `eps` and a count `min_samples`. A point with at least `min_samples` points within
`eps` (itself included) is a **core** point. Core points that are within `eps` of each other share a cluster;
points near a core point join as **border** points; everything else is **noise** (label `-1`).

In [ ]:
from sklearn.cluster import DBSCAN

rng = np.random.default_rng(0)
Xn = np.vstack([Xm, rng.uniform([-1.5, -1], [2.5, 1.5], size=(25, 2))])   # moons plus 25 stray points
db = DBSCAN(eps=0.2, min_samples=5).fit(Xn)
lab = db.labels_
core = np.zeros(len(Xn), bool)
core[db.core_sample_indices_] = True
print(f"clusters: {lab.max() + 1}, core points: {core.sum()}, border points: {(~core & (lab >= 0)).sum()}, noise: {(lab == -1).sum()}")

plt.figure(figsize=(6.4, 3.8))
plt.scatter(*Xn[core].T, c=palette[lab[core]], s=28, label="core")
plt.scatter(*Xn[~core & (lab >= 0)].T, c=palette[lab[~core & (lab >= 0)]], s=10, label="border")
plt.scatter(*Xn[lab == -1].T, c="#8a85a3", marker="x", s=24, label="noise")
plt.legend(loc="upper right")
plt.title("DBSCAN follows the moons and leaves the strays out")
plt.show()

## Choosing eps: the k-distance plot
For every point, find the distance to its `min_samples`-th nearest neighbor and sort them. Points inside
crowds have small distances, strays large ones. A good `eps` sits at the bend, where the curve shoots up.

In [ ]:
from sklearn.neighbors import NearestNeighbors

dist, _ = NearestNeighbors(n_neighbors=5).fit(Xn).kneighbors(Xn)   # includes the point itself, like min_samples
kdist = np.sort(dist[:, -1])
plt.figure(figsize=(6.4, 3.4))
plt.plot(kdist, color="#7b5cfa")
plt.axhline(0.2, color="#2c2842", ls="--", lw=1.2)
plt.text(5, 0.22, "eps = 0.2", color="#2c2842")
plt.xlabel("points, sorted")
plt.ylabel("distance to 5th neighbor")
plt.title("Crowded points on the left, strays on the right")
plt.show()

## Crowds of different density: HDBSCAN
One radius can't fit a tight crowd and a loose one at the same time. **HDBSCAN** tries every radius at once
(it builds a hierarchy of DBSCAN results) and keeps the clusters that last longest. You only choose the
smallest cluster size you care about.

In [ ]:
from sklearn.cluster import HDBSCAN

Xd, _ = make_blobs(n_samples=[150, 150, 60], centers=[[0, 0], [5, 0], [2.5, 4]], cluster_std=[0.3, 1.2, 0.5], random_state=3)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True, layout="constrained")
for ax, (name, model) in zip(axes, [("DBSCAN, eps = 0.3", DBSCAN(eps=0.3, min_samples=5)),
                                     ("DBSCAN, eps = 1.0", DBSCAN(eps=1.0, min_samples=5)),
                                     ("HDBSCAN", HDBSCAN(min_cluster_size=15, copy=True))]):
    lab = model.fit_predict(Xd)
    ax.scatter(Xd[:, 0], Xd[:, 1], c=np.where(lab >= 0, palette[lab % 8], "#d9d5e6"), s=10)
    ax.set_title(f"{name}: {lab.max() + 1} clusters, {(lab == -1).sum()} noise")
plt.show()

## Your turn: sweep eps
Run DBSCAN on `Xn` (the moons with strays) with `min_samples=5` and `eps` from 0.05 to 0.5.
For each, print the number of clusters and of noise points. Which range of eps gives the two moons?

In [ ]:
for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    lab = None   # TODO: DBSCAN(eps=eps, min_samples=5).fit_predict(Xn)
    print(f"eps {eps:.2f}: clusters {None}, noise points {None}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xn)
    print(f"eps {eps:.2f}: clusters {lab.max() + 1}, noise points {(lab == -1).sum()}")
print("Only eps = 0.2 finds the two moons. Smaller: noise and fragments. Bigger: the moons and strays melt into one blob.")